# 04 — Interpretable anomaly models

**Outcome:** fit calibration-frozen self references, add peer and group
evidence when topology is available, and select a small predeclared
portfolio on development data at a matched false-incident workload.

The model never reads SPEC-EVAL. Evaluation truth is loaded only after
score and incident construction, through the frozen harness output.


## 1. Setup


In [ ]:
import importlib
import os
import sys
import tempfile
import time
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT") or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_11_run1",
    "petrobras_3w": "petrobras_3w_core_v0_11_run1",
}
from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json
import evaluation_core, simple_model_core
evaluation_core = importlib.reload(evaluation_core)
simple_model_core = importlib.reload(simple_model_core)
from evaluation_core import evaluate_cases, form_cases
from simple_model_core import (
    MODEL_CORE_VERSION, alert_grid_from_score_file, calibration_thresholds,
    case_score_trace, duration_to_observations, fit_residual_bundle,
    fit_topology_reference, materialize_measurement_features,
    materialize_wide_partition, merge_score_files, partition_exposure,
    score_residual_file, score_topology_file,
)

RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / os.getenv(
    "CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR]
)
CORE_ROOT, SPLIT_ROOT = RUN_ROOT / "SPEC-CORE", RUN_ROOT / "SPLITS"
VERSION = "3.0.0"
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{VERSION}" / SECTOR / f"{SECTOR}_eda_v3_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{VERSION}" / SECTOR / f"{SECTOR}_evaluation_v3_run1"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{VERSION}" / SECTOR / f"{SECTOR}_models_v3_run1"
AUDIT_ROOT = DATA_ROOT / "outputs" / "audits" / "v1.0.0" / "telecom" / "telecom_localisation_audit_run1"
MAX_TRAINING_ROWS = int(os.getenv("MODEL_MAX_TRAINING_ROWS", "150000"))

manifest = read_json(CORE_ROOT / "manifest.json")
decisions = read_json(EDA_ROOT / "eda_decisions.json")
policy = read_json(EVALUATION_ROOT / "evaluation_policy.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
baseline_review = pd.read_parquet(EDA_ROOT / "baseline_review.parquet")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.is_file() else pd.DataFrame()
if decisions["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("EDA and canonical input do not match")
if policy["canonical_fingerprint"] != manifest["fingerprint"]:
    raise ValueError("Evaluation and canonical input do not match")
display(pd.Series({
    "sector": SECTOR, "topology": not topology.empty,
    "training_rows_cap": MAX_TRAINING_ROWS, "output": MODEL_ROOT,
}, name="value").to_frame())


## 2. Fit frozen self-history residuals


In [ ]:
started = time.perf_counter()
temporary = tempfile.TemporaryDirectory()
work = Path(temporary.name)
paths = {}
lookback_seconds = max(
    decisions["dispersion_window_seconds"], decisions["base_cadence_seconds"]
)

for partition in ("calibration", "development"):
    wide = work / f"{partition}_wide.parquet"
    features = work / f"{partition}_features.parquet"
    split = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, partition, catalogue, wide,
        lookback_seconds=lookback_seconds,
    )
    materialize_measurement_features(
        wide, catalogue, features,
        target_cadence_seconds=decisions["base_cadence_seconds"],
        score_start=split["score_start"] if partition == "calibration" else None,
        score_end=split["score_end"] if partition == "calibration" else None,
    )
    paths[partition] = {"wide": wide, "features": features, **split}

bundle = fit_residual_bundle(
    paths["calibration"]["features"],
    use_entity_reference=decisions["primary_split"] == "time",
    reference_exclusions=baseline_review.loc[
        baseline_review.review_flag, ["entity_id", "metric_id"]
    ],
    maximum_training_rows=MAX_TRAINING_ROWS,
    fit_multivariate=False,
)

for partition in paths:
    self_scores = work / f"{partition}_self_scores.parquet"
    residuals = work / f"{partition}_residuals.parquet"
    score_residual_file(
        bundle, paths[partition]["features"], self_scores,
        cadence_seconds=decisions["base_cadence_seconds"],
        dispersion_window_seconds=decisions["dispersion_window_seconds"],
        cusum_allowance=policy["cusum_allowance"],
        residual_destination=residuals,
        score_start=paths[partition]["score_start"],
        score_end=paths[partition]["score_end"],
    )
    paths[partition].update(self_scores=self_scores, residuals=residuals, scores=self_scores)

print(f"Self-history fit and scoring: {(time.perf_counter() - started) / 60:.1f} minutes")


## 3. Add peer and common-mode evidence when topology is available


In [ ]:
topology_reference = pd.DataFrame()
topology_decisions = {}
if not topology.empty:
    decision_path = AUDIT_ROOT / "topology_decisions.json"
    if not decision_path.is_file():
        raise FileNotFoundError("Run 00_TELECOM_LOCALISATION_AUDIT.ipynb first")
    topology_decisions = read_json(decision_path)
    peer_level = topology_decisions["primary_peer_level"]
    if not topology_decisions["peer_channel_enabled"] or not peer_level:
        raise ValueError("Phase 0 did not approve a peer-comparison level")
    group_levels = topology_decisions["common_mode_levels"]
    topology_reference = fit_topology_reference(
        paths["calibration"]["residuals"], topology, bundle["feature_columns"],
        peer_group_type=peer_level, group_types=group_levels,
        min_peers=topology_decisions["minimum_valid_peers"],
        min_group_entities=topology_decisions["minimum_group_entities"],
        min_group_fraction=topology_decisions["minimum_group_available_fraction"],
    )
    for partition in paths:
        topology_scores = work / f"{partition}_topology_scores.parquet"
        combined = work / f"{partition}_combined_scores.parquet"
        score_topology_file(
            paths[partition]["residuals"], topology, topology_reference,
            topology_scores, peer_group_type=peer_level,
            group_types=group_levels,
            min_peers=topology_decisions["minimum_valid_peers"],
            min_group_entities=topology_decisions["minimum_group_entities"],
            min_group_fraction=topology_decisions["minimum_group_available_fraction"],
        )
        merge_score_files(paths[partition]["self_scores"], topology_scores, combined)
        paths[partition]["scores"] = combined

with duckdb.connect() as connection:
    score_columns = set(connection.execute(
        "SELECT * FROM read_parquet(?) LIMIT 0", [str(paths["calibration"]["scores"])]
    ).df().columns)
    active_channels = [channel for channel in policy["selection_channels"] if channel in score_columns]
    counts = connection.execute(
        "SELECT " + ", ".join(
            f"count({channel}) AS {channel}" for channel in active_channels
        ) + " FROM read_parquet(?)",
        [str(paths["calibration"]["scores"])],
    ).df().iloc[0]
unavailable = [channel for channel in active_channels if counts[channel] == 0]
if unavailable:
    raise ValueError(f"Approved channels have no calibration evidence: {unavailable}")
display(topology_reference)
display(counts.rename("calibration_scores").to_frame())


## 4. Compare a small predeclared portfolio set


In [ ]:
block_column = "entity_id" if decisions["primary_split"] == "time" else "episode_id"
thresholds = calibration_thresholds(
    paths["calibration"]["scores"], policy["threshold_quantiles"],
    block_column=block_column,
    block_duration_seconds=policy["calibration_block_seconds"],
    minimum_block_rows=20,
    model_ids=active_channels,
)
cadence = decisions["base_cadence_seconds"]
persistence = {
    channel: duration_to_observations(
        policy["channel_persistence_seconds"][channel], cadence
    ) for channel in active_channels
}
recovery = duration_to_observations(policy["recovery_seconds"], cadence)
alert_grid = alert_grid_from_score_file(
    paths["development"]["scores"], thresholds,
    persistence=persistence, recovery_consecutive=recovery,
)

portfolios = {
    "rapid_only": ["rapid_residual"],
    "self_history": ["rapid_residual", "drift_cusum"],
}
if {"peer_deviation", "group_common_mode"} <= set(active_channels):
    portfolios.update({
        "self_plus_peer": ["rapid_residual", "drift_cusum", "peer_deviation"],
        "self_peer_group": ["rapid_residual", "drift_cusum", "peer_deviation", "group_common_mode"],
    })
if "dispersion_change" in active_channels:
    portfolios["full_with_dispersion"] = [*portfolios[list(portfolios)[-1]], "dispersion_change"]

def threshold_value(channel, quantile):
    row = thresholds.loc[
        thresholds.model_id.eq(channel)
        & thresholds.threshold_quantile.eq(float(quantile)), "threshold"
    ]
    return float(row.iloc[0])

def alerts_for(channels, quantile):
    frames = [alert_grid[(channel, float(quantile))] for channel in channels]
    frames = [frame for frame in frames if not frame.empty]
    channel_thresholds = {
        channel: threshold_value(channel, quantile) for channel in channels
    }
    if not frames:
        from evaluation_core import ALERT_COLUMNS
        return pd.DataFrame(columns=ALERT_COLUMNS), channel_thresholds
    alerts = pd.concat(frames, ignore_index=True).sort_values("alert_start").reset_index(drop=True)
    alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    return alerts, channel_thresholds

# Scoring is complete before evaluation truth is loaded.
fault_events = pd.read_parquet(
    EVALUATION_ROOT / "development" / "fault_events.parquet"
)
fault_intervals = pd.read_parquet(
    EVALUATION_ROOT / "development" / "fault_entity_intervals.parquet"
)
exposure = partition_exposure(paths["development"]["scores"], policy["exposure_unit"], cadence)
horizon = policy.get("decision_horizon_seconds_by_fault_type") or policy["decision_horizon_seconds"]
comparisons = []
for portfolio, channels in portfolios.items():
    for quantile in policy["threshold_quantiles"]:
        alerts, channel_thresholds = alerts_for(channels, quantile)
        cases, members = form_cases(
            alerts, topology, gap_seconds=policy["case_gap_seconds"],
            thresholds=channel_thresholds,
        )
        result = evaluate_cases(
            cases, members, fault_events, fault_intervals,
            exposure_value=exposure, exposure_unit=policy["exposure_unit"],
            decision_horizon_seconds=horizon, topology_memberships=topology,
        )
        metrics = result["metrics"].set_index("metric")
        false_metric = f"false_cases_per_{policy['exposure_unit']}"
        key = f"{portfolio}|q={quantile}"
        comparisons.append({
            "candidate_key": key, "portfolio": portfolio,
            "threshold_quantile": quantile, "channels": ", ".join(channels),
            "event_recall": metrics.at["event_recall", "value"],
            "event_recall_ci_low": metrics.at["event_recall", "ci_low"],
            "event_recall_ci_high": metrics.at["event_recall", "ci_high"],
            "false_case_rate": metrics.at[false_metric, "value"],
            "false_case_rate_ci_high": metrics.at[false_metric, "ci_high"],
            "case_precision": metrics.at["case_precision", "value"],
            "joint_detection_localisation": metrics.at[
                "joint_detection_and_localisation_recall", "value"
            ],
            "median_delay_seconds": metrics.at["median_detection_delay_seconds", "value"],
            "cases": len(cases),
        })
        print(f"Evaluated {key}: {len(cases):,} cases")
comparison = pd.DataFrame(comparisons)
display(comparison.sort_values(["portfolio", "threshold_quantile"]))


## 5. Select on development and save the evidence


In [ ]:
budget_gate = policy["false_case_budget"] * policy["budget_safety_factor"]
eligible = comparison.loc[comparison.false_case_rate_ci_high.le(budget_gate)].copy()
selection_status = "within_budget"
if eligible.empty:
    eligible = comparison.nsmallest(1, "false_case_rate_ci_high").copy()
    selection_status = "no_configuration_within_budget"

best_recall = eligible.event_recall.max()
equivalent = eligible.loc[
    eligible.event_recall.ge(best_recall - policy["equivalence_margin"])
].copy()
preference = {name: rank for rank, name in enumerate(portfolios)}
equivalent["preference"] = equivalent.portfolio.map(preference)
selected = equivalent.sort_values([
    "preference", "false_case_rate_ci_high", "median_delay_seconds"
]).iloc[0]
selected_channels = portfolios[selected.portfolio]
alerts, selected_thresholds = alerts_for(
    selected_channels, float(selected.threshold_quantile)
)
cases, members = form_cases(
    alerts, topology, gap_seconds=policy["case_gap_seconds"],
    thresholds=selected_thresholds,
)
result = evaluate_cases(
    cases, members, fault_events, fault_intervals,
    exposure_value=exposure, exposure_unit=policy["exposure_unit"],
    decision_horizon_seconds=horizon, topology_memberships=topology,
)
cases = cases.sort_values("anomaly_evidence_score", ascending=False).reset_index(drop=True)
cases.insert(0, "rank", np.arange(1, len(cases) + 1))
trace = case_score_trace(
    paths["development"]["scores"], paths["development"]["features"],
    cases, members, alerts, bundle, selected_thresholds, selected_channels,
    window_seconds=max(decisions["dispersion_window_seconds"], 20 * cadence),
)

configuration = {
    "model_version": VERSION,
    "model_core_version": MODEL_CORE_VERSION,
    "sector": SECTOR,
    "canonical_fingerprint": manifest["fingerprint"],
    "portfolio": selected.portfolio,
    "channels": selected_channels,
    "threshold_quantile": float(selected.threshold_quantile),
    "thresholds": selected_thresholds,
    "threshold_method": "calibration block maxima",
    "selection_status": selection_status,
    "false_case_budget": policy["false_case_budget"],
    "budget_gate": budget_gate,
    "exposure_unit": policy["exposure_unit"],
    "case_gap_seconds": policy["case_gap_seconds"],
    "channel_persistence_seconds": policy["channel_persistence_seconds"],
    "channel_persistence_observations": persistence,
    "recovery_seconds": policy["recovery_seconds"],
    "recovery_observations": recovery,
    "cusum_allowance": policy["cusum_allowance"],
    "decision_horizon_seconds": policy["decision_horizon_seconds"],
    "decision_horizon_seconds_by_fault_type": policy.get("decision_horizon_seconds_by_fault_type", {}),
    "feature_settings": decisions,
    "lookback_seconds": lookback_seconds,
    "topology_enabled": not topology.empty,
    "topology_decisions": topology_decisions,
    "synthetic_limit": policy["synthetic_limit"],
    "holdout_used": False,
}

with new_output_directory(MODEL_ROOT) as output:
    joblib.dump(bundle, output / "residual_bundle.joblib")
    write_json(output / "selected_configuration.json", configuration)
    thresholds.to_csv(output / "calibration_thresholds.csv", index=False)
    comparison.to_csv(output / "development_comparison.csv", index=False)
    topology_reference.to_csv(output / "topology_reference.csv", index=False)
    alerts.to_parquet(output / "selected_alerts.parquet", index=False)
    cases.to_parquet(output / "selected_cases.parquet", index=False)
    members.to_parquet(output / "selected_case_members.parquet", index=False)
    trace.to_parquet(output / "development_case_trace.parquet", index=False)
    result["metrics"].to_csv(output / "development_metrics.csv", index=False)
    result["fault_type_results"].to_csv(output / "fault_type_results.csv", index=False)
    result["domain_type_results"].to_csv(output / "domain_type_results.csv", index=False)
    result["localisation_results"].to_csv(output / "localisation_results.csv", index=False)

display(pd.Series(configuration, name="selected").to_frame())
display(result["metrics"])
if selection_status != "within_budget":
    print("STOP — no configuration met the development workload gate; keep holdout sealed")
print("Saved:", MODEL_ROOT)
print("Next: 05_INCIDENT_RANKING_AND_HOLDOUT.ipynb")
temporary.cleanup()
